# CT-Objekt-Composer

Interaktives Jupyter-GUI fuer industrielle CT-/Roentgenbilder: Objekte bekannter Groesse
(z.B. 50 um) aus einem Bild ausschneiden, in ein anderes Bild einsetzen und pruefen, ab welcher
Groesse eine Bildauswertung sie noch findet.

**Start:** beide Code-Zellen von oben nach unten ausfuehren. Ohne eigene Daten laeuft alles mit
dem eingebauten Demo-CT, es werden keine weiteren Dateien gebraucht.

**Benoetigt:** `numpy`, `scipy`, `pillow`, `matplotlib`, `ipywidgets`, `ipympl`
(in Anaconda alles dabei, sonst: `pip install numpy scipy pillow matplotlib ipywidgets ipympl`).
`ipympl` liefert die interaktiven Bildflaechen &ndash; ohne das Paket gibt es keine Maussteuerung.

---

### 1 - Ausschneiden

CT laden (Pfad, Upload oder Demo-Knopf), dann mit der **linken Maustaste im Bild einen Rahmen um
das Objekt ziehen**. Ecken und Kanten des Rahmens bleiben danach verschiebbar. Ist in der
Matplotlib-Toolbar Zoom oder Pan aktiv, geht das Ziehen dorthin &ndash; Werkzeug abwaehlen.

- `um / Pixel` steuert alle Groessenangaben (um und mm).
- `Grauwert-Fenster` ist die Darstellung, kein Eingriff in die Daten: absolute Grauwerte 0..1.
  Eng ziehen macht schwache Strukturen sichtbar. `Auto-Fenster` holt sich passende Grenzen
  aus dem geladenen CT.
- **Ausschnitt exportieren** legt ihn als 16-bit PNG in `patches/` ab, **als Objekt uebernehmen**
  macht ihn direkt zum aktiven Objekt fuer Schritt 2.

### 2 - Platzieren

**Klick auf freie Flaeche legt das aktive Objekt dort ab** &ndash; beliebig oft, auch mit
verschiedenen Objekten aus der Liste. **Klick auf ein Objekt greift es, Ziehen verschiebt es.**
Alle abgelegten Objekte stehen unter *Objekte im Bild*, das aktive ist mit einem Pfeil markiert
und gelb umrahmt. Die Regler wirken jeweils auf das aktive Objekt:

- Groesse in px **oder direkt in um** (beide Felder sind gekoppelt).
- `Feather-Kante`, `Kontrast (Gain)`, `Ellipsen-Maske` fuer den Uebergang.
- `Einblendung`: *Grauwerte behalten* laesst das Objekt als Koerper sichtbar;
  *Rand an Hintergrund anpassen* schiebt den Patchrand auf das lokale Hintergrundniveau,
  dann verschwindet der Uebergang (fuer Fehlstellen mit Materialkontext).

**Der Hintergrund ist frei waehlbar:** das leere Detektorbild (Pixelzahl, Grauwert, Rauschen
einstellbar) ist nur der Platzhalter &ndash; ueber *Bild laden* oder *Upload* kommt ein eigenes
Bild rein und wird 1:1 uebernommen, damit um/Pixel weiter stimmt. `Hintergrund an Objekt anpassen`
setzt Pegel und Rauschen des leeren Bildes auf den Rand des Objekts, damit kein Patch-Quadrat
sichtbar wird. Export: 16-bit PNG nach `scenes/`.

Die Statuszeile zeigt fuer das aktive Objekt **Kontrast und SNR** gegen den Hintergrund &ndash;
die Kennzahl dafuer, ob eine Auswertung es ueberhaupt finden kann.

---

### Demo-CT

Roentgenbild einer M4-Schraube aus Stahl: der Grauwert kodiert die durchstrahlte Materialdicke
(Beer-Lambert mit `MU_STEEL`), dazu Detektorunschaerfe (`BLUR_UM`) und Photonenrauschen
(`N_PHOTONS`) &ndash; alle drei sind Stellschrauben oben in der ersten Code-Zelle.
Enthalten: Gewinde, Kopf mit Innensechskant, drei Poren (300/120/60 um) und vier Kalibrierkugeln
mit **50 / 100 / 200 / 400 um** Durchmesser (`DEMO_BALLS`) als fertige Testobjekte. Beim Start
liegt der Rahmen bereits auf der 400-um-Kugel.

Gemessen bei 10 um/px gegen den Luftbereich (0.0028 +- 0.0041): 50 um -> SNR 3.2,
100 um -> 7.1, 200 um -> 13.2, 400 um -> 23.0.

### Anpassen

`VIEW` (erste Zeile der GUI-Zelle) ist die Kantenlaenge der Bildflaechen in Pixeln &ndash;
kleiner setzen, wenn der Bildschirm knapp ist, groesser fuer mehr Details.

In [7]:
# --- Kernfunktionen ---
import io, numpy as np
from pathlib import Path
from scipy.ndimage import gaussian_filter
from PIL import Image, ImageDraw
import ipywidgets as w
from IPython.display import display

PATCHES = Path('patches'); PATCHES.mkdir(exist_ok=True)
SCENES = Path('scenes'); SCENES.mkdir(exist_ok=True)

MU_STEEL = 0.20        # effektiver Schwaechungskoeffizient Stahl [1/mm], ca. 150 kV gefiltert
                       # (klein genug, dass dickes Material nicht in Weiss saettigt)
N_PHOTONS = 20000      # Photonen pro Pixel -> Rauschniveau (kleiner = mehr Rauschen)
BLUR_UM = 12.0         # Detektor-/Brennfleckunschaerfe [um]
# Demo-Bauteil: Kalibrierkugeln (x [mm], y [mm], Durchmesser [um]) neben der Schraube
DEMO_BALLS = ((9.0, -6.0, 50), (9.0, -2.0, 100), (9.0, 2.0, 200), (9.0, 6.0, 400))
DEMO_PORES = ((0.0, -2.0, 300), (0.8, 3.0, 120), (-0.6, 6.5, 60))   # relativ zur Schraubenachse
SCREW_X = -4.0         # Schraubenachse liegt 4 mm links der Bildmitte


def load_gray(src):
    'Pfad oder bytes -> float32 Graubild 0..1 (8/16 bit und float werden erkannt)'
    im = Image.open(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else str(src))
    if im.mode not in ('I;16', 'I;16B', 'I', 'F', 'L'):
        im = im.convert('L')
    a = np.asarray(im).astype(np.float32)
    if a.ndim == 3:
        a = a.mean(2)
    m = float(a.max())
    d = 1.0 if m <= 1.0 else (255.0 if m <= 255.0 else 65535.0)
    return a / d


def save16(img, path):
    Image.fromarray((np.clip(img, 0, 1) * 65535).astype(np.uint16)).save(str(path))


def blur(a, sigma):
    if sigma <= 0:
        return a
    return gaussian_filter(a, sigma, mode='nearest').astype(np.float32)


def mm2px(x_mm, y_mm, n, um_px):
    'mm-Koordinate (0,0 = Bildmitte) -> Pixelindex (jx, jy)'
    return int(round(x_mm * 1000 / um_px + n / 2)), int(round(y_mm * 1000 / um_px + n / 2))


def add_sphere(d, cx_mm, cy_mm, dia_um, um_px, sign=1):
    '''Kugel in die Dickenkarte d [mm] einrechnen. Supersampling, damit auch Kugeln kleiner
    als ein Pixel volumenrichtig ankommen - genau der 50-um-Testfall.'''
    n = d.shape[0]
    mm = um_px / 1000.0
    r = dia_um / 2000.0
    ss = int(np.clip(np.ceil(3 * mm / max(2 * r, 1e-9)), 4, 32))
    j0 = max(0, int(np.floor((cx_mm - r) / mm + n / 2)) - 1)
    j1 = min(n, int(np.ceil((cx_mm + r) / mm + n / 2)) + 2)
    i0 = max(0, int(np.floor((cy_mm - r) / mm + n / 2)) - 1)
    i1 = min(n, int(np.ceil((cy_mm + r) / mm + n / 2)) + 2)
    if j1 <= j0 or i1 <= i0:
        return
    off = (np.arange(ss) + 0.5) / ss - 0.5
    xs = ((np.arange(j0, j1)[:, None] + off[None, :]).ravel() - n / 2) * mm - cx_mm
    ys = ((np.arange(i0, i1)[:, None] + off[None, :]).ravel() - n / 2) * mm - cy_mm
    t = 2 * np.sqrt(np.maximum(r * r - np.add.outer(ys * ys, xs * xs), 0))
    d[i0:i1, j0:j1] += sign * t.reshape(i1 - i0, ss, j1 - j0, ss).mean((1, 3)).astype(np.float32)


def demo_ct(n=3000, um_px=10.0, seed=0):
    '''Roentgenbild einer M4-Schraube aus Stahl: der Grauwert kodiert die durchstrahlte
    Materialdicke (Beer-Lambert), dazu Detektorunschaerfe und Photonenrauschen.
    Drin: Gewinde, Kopf mit Innensechskant, drei Poren und vier Kalibrierkugeln
    mit 50 / 100 / 200 / 400 um Durchmesser (siehe DEMO_BALLS).'''
    mm = um_px / 1000.0
    X = ((np.arange(n, dtype=np.float32) - n / 2) * mm)[None, :] - SCREW_X   # 0 = Schraubenachse
    Y = ((np.arange(n, dtype=np.float32) - n / 2) * mm)[:, None]
    d = np.zeros((n, n), np.float32)
    rh = 3.5                                                    # Kopf D 7 mm, 3 mm hoch
    d += np.where((Y >= -10.5) & (Y < -7.5) & (np.abs(X) < rh),
                  2 * np.sqrt(np.maximum(rh * rh - X * X, 0)), 0)
    d -= np.where((Y >= -10.4) & (Y < -8.2) & (np.abs(X) < 1.3), 2.0, 0)   # Innensechskant, vereinfacht
    r = 1.6 + 0.4 * (0.5 + 0.5 * np.cos(2 * np.pi * Y / 0.7))    # Gewinde, Steigung 0.7 mm
    r = r * np.clip((10.5 - Y) / 2.0, 0, 1)                      # angeschraegte Spitze
    d += np.where((Y >= -7.5) & (Y < 10.5) & (np.abs(X) < r),
                  2 * np.sqrt(np.maximum(r * r - X * X, 0)), 0)
    for cx, cy, dia in DEMO_PORES:
        add_sphere(d, SCREW_X + cx, cy, dia, um_px, sign=-1)
    for cx, cy, dia in DEMO_BALLS:
        add_sphere(d, cx, cy, dia, um_px)
    T = np.exp(-MU_STEEL * np.maximum(d, 0))                     # Transmission
    T = blur(T, max(0.5, BLUR_UM / um_px))
    rng = np.random.default_rng(seed)
    T += np.sqrt(np.maximum(T, 1e-6) / N_PHOTONS) * rng.standard_normal(T.shape).astype(np.float32)
    return np.clip(1.0 - T, 0, 1).astype(np.float32)             # viel Material = hell


def clamp_roi(roi, shape):
    'ROI (x0,y0,x1,y1) auf Bildgrenzen begrenzen und sortieren'
    h, wd = shape
    x0, x1 = sorted((int(round(roi[0])), int(round(roi[2]))))
    y0, y1 = sorted((int(round(roi[1])), int(round(roi[3]))))
    return (max(0, min(x0, wd - 2)), max(0, min(y0, h - 2)),
            min(wd, max(x1, x0 + 2)), min(h, max(y1, y0 + 2)))


def crop_roi(img, roi):
    x0, y0, x1, y1 = clamp_roi(roi, img.shape)
    return img[y0:y1, x0:x1]


def resize(img, size):
    'skaliert so, dass die laengste Kante = size ist (Seitenverhaeltnis bleibt)'
    h, wd = img.shape
    k = size / max(h, wd)
    tw, th = max(2, int(round(wd * k))), max(2, int(round(h * k)))
    return np.asarray(Image.fromarray(np.ascontiguousarray(img, np.float32), mode='F')
                      .resize((tw, th), Image.LANCZOS)).astype(np.float32)


def feather_mask(h, wd, px, ellipse=False):
    'weiche Alpha-Maske: smoothstep-Rampe am Rand, optional zusaetzlich elliptisch'
    def ramp(n):
        v = np.ones(n, np.float32)
        p = int(min(px, n // 2))
        if p > 0:
            t = (np.arange(p) + 0.5) / p
            s = (t * t * (3 - 2 * t)).astype(np.float32)
            v[:p] = s
            v[n - p:] = s[::-1]
        return v
    m = np.minimum.outer(ramp(h), ramp(wd)).astype(np.float32)
    if ellipse:
        yy = (np.arange(h) + 0.5) / h * 2 - 1
        xx = (np.arange(wd) + 0.5) / wd * 2 - 1
        rr = np.sqrt(np.add.outer(yy * yy, xx * xx)).astype(np.float32)
        band = max(1e-3, 2.0 * max(px, 1) / max(h, wd))
        e = np.clip((1.0 - rr) / band, 0, 1)
        m = m * (e * e * (3 - 2 * e))
    return m


def rim_mean(a):
    'Mittelwert des aeussersten Rings - der Grauwert, der an den Hintergrund stoesst'
    if min(a.shape) <= 2:
        return float(a.mean())
    return float(np.concatenate([a[0], a[-1], a[1:-1, 0], a[1:-1, -1]]).mean())


def blend(canvas, patch, cx, cy, feather=16, match_level=False, gain=1.0, ellipse=False):
    '''Patch mittig auf (cx, cy) einblenden (in-place). Rueckgabe: bbox (x0,y0,x1,y1) oder None.
    match_level=False: Grauwerte des Objekts bleiben erhalten (sichtbares Objekt auf leerem Feld).
    match_level=True:  Patchrand wird auf das lokale Hintergrundniveau geschoben (nahtlos im Material).
    gain skaliert den Kontrast um den Randgrauwert.'''
    H, W = canvas.shape
    h, wd = patch.shape
    y0, x0 = int(cy) - h // 2, int(cx) - wd // 2
    ys0, xs0 = max(0, y0), max(0, x0)
    ys1, xs1 = min(H, y0 + h), min(W, x0 + wd)
    if ys1 <= ys0 or xs1 <= xs0:
        return None
    sub = patch[ys0 - y0:ys1 - y0, xs0 - x0:xs1 - x0]
    m = feather_mask(h, wd, feather, ellipse)[ys0 - y0:ys1 - y0, xs0 - x0:xs1 - x0]
    dst = canvas[ys0:ys1, xs0:xs1]
    ref = rim_mean(sub)
    base = float(dst.mean()) if match_level else ref
    src = (sub - ref) * gain + base
    canvas[ys0:ys1, xs0:xs1] = np.clip(dst * (1 - m) + src * m, 0, 1)
    return (xs0, ys0, xs1, ys1)


def make_canvas(shape, level=0.003, noise=0.004, seed=42):
    '''leeres Detektorbild: Grauwert + Rauschen wie im Luftbereich einer Aufnahme.
    shape: Kantenlaenge (quadratisch) oder (hoehe, breite).'''
    h, wd = (shape, shape) if isinstance(shape, (int, np.integer)) else shape
    rng = np.random.default_rng(seed)
    a = level + noise * rng.standard_normal((h, wd)).astype(np.float32)
    return np.clip(a, 0, 1).astype(np.float32)


def to_pil(img, maxsize=560, lo=0.0, hi=1.0):
    '''Grauwertfenster + Downscale -> 8-bit RGB fuer die Anzeige.
    lo/hi sind ABSOLUTE Grauwerte (0..1), keine Perzentile: in einem fast leeren Bild
    liegt das 99.5-Perzentil im Rauschen und die Anzeige zeigt nur noch Rauschen.'''
    a = np.ascontiguousarray(img, np.float32)
    s = 1.0
    if max(a.shape) > maxsize:
        s = maxsize / max(a.shape)
        a = np.asarray(Image.fromarray(a, mode='F').resize(
            (max(1, int(round(a.shape[1] * s))), max(1, int(round(a.shape[0] * s)))), Image.BILINEAR))
    a = np.clip((a - lo) / max(1e-6, hi - lo), 0, 1)
    return Image.fromarray((a * 255).astype(np.uint8)).convert('RGB'), s


def png(img, maxsize=560, boxes=(), lo=0.0, hi=1.0):
    'PNG-bytes fuer ipywidgets.Image, mit optionalen Rahmen'
    im, s = to_pil(img, maxsize, lo, hi)
    if boxes:
        d = ImageDraw.Draw(im)
        for (x0, y0, x1, y1) in boxes:
            d.rectangle([x0 * s, y0 * s, max(x0 * s, x1 * s - 1), max(y0 * s, y1 * s - 1)],
                        outline=(255, 90, 70), width=2)
    b = io.BytesIO(); im.save(b, 'PNG')
    return b.getvalue()


def _selfcheck():
    n, upx = 256, 240.0
    a = demo_ct(n, upx, 1)
    assert a.shape == (n, n)
    bg = float(a[:20, :20].mean())
    hx, hy = mm2px(SCREW_X, -9.0, n, upx)
    sx, sy = mm2px(SCREW_X, 0.0, n, upx)
    head = float(a[hy - 3:hy + 4, hx - 5:hx + 6].mean())
    shaft = float(a[sy - 8:sy + 8, sx - 3:sx + 4].mean())
    assert head > shaft > bg + 0.3, (head, shaft, bg)       # Grauwert steigt mit Materialdicke
    bx, by = mm2px(9.0, 6.0, n, upx)                        # 400-um-Kalibrierkugel
    air = float(a[:20, -20:].std())
    sig = float(a[by - 3:by + 4, bx - 3:bx + 4].max()) - bg
    assert sig > 3 * air, (sig, air)                        # hebt sich vom Rauschen ab
    assert float(a[:20, -20:].std()) < 0.05                 # Luftbereich ist ruhig
    # Ausschnitt / ROI
    assert crop_roi(a, (10, 20, 60, 100)).shape == (80, 50)
    assert crop_roi(a, (60, 100, 10, 20)).shape == (80, 50)          # verdreht gezogen
    assert clamp_roi((-50, -50, 999, 999), a.shape) == (0, 0, n, n)
    assert resize(np.zeros((100, 50), np.float32), 64).shape == (64, 32)   # Seitenverhaeltnis
    # Einblenden
    c = crop_roi(a, (sx - 25, sy - 25, sx + 25, sy + 25))
    assert blend(make_canvas(300, 0.1, 0.0), c, 150, 150, 8) == (125, 125, 175, 175)
    assert blend(make_canvas(50), c, -500, -500) is None              # komplett ausserhalb
    assert blend(make_canvas(300), c, 5, 5) == (0, 0, 30, 30)         # Clipping am Rand
    hard = make_canvas(300, 0.1, 0.0); blend(hard, c, 150, 150, 0)
    soft = make_canvas(300, 0.1, 0.0); blend(soft, c, 150, 150, 20)
    assert abs(hard[0, 0] - 0.1) < 1e-6                              # Hintergrund unberuehrt
    assert float(hard[140:160, 140:160].mean()) > 0.3                # Grauwerte bleiben erhalten
    eh = float(np.abs(hard[125, 125:175] - 0.1).mean())
    es = float(np.abs(soft[125, 125:175] - 0.1).mean())
    assert es < 0.5 * eh                                             # Feather glaettet die Kante
    seam = make_canvas(300, 0.1, 0.0); blend(seam, c, 150, 150, 0, match_level=True)
    ring = np.concatenate([seam[125, 125:175], seam[174, 125:175],
                           seam[125:175, 125], seam[125:175, 174]])
    assert abs(float(ring.mean()) - 0.1) < 0.02       # Patchrand liegt im Mittel auf bg-Niveau
    # Anzeige-Fenster rechnet mit absoluten Grauwerten
    lin = np.linspace(0, 1, 256, dtype=np.float32)[None, :].repeat(4, 0)
    assert np.asarray(to_pil(lin, 999, 0.0, 1.0)[0])[0, -1, 0] == 255
    assert np.asarray(to_pil(lin, 999, 0.0, 0.5)[0])[0, 128, 0] == 255   # enges Fenster saettigt
    assert np.asarray(to_pil(lin, 999, 0.5, 1.0)[0])[0, 0, 0] == 0
    # leeres Bild bleibt dunkel, ein eingesetztes Objekt hebt sich ab
    assert make_canvas((30, 50)).shape == (30, 50)        # rechteckiger Hintergrund
    cv = make_canvas(200)
    assert np.asarray(to_pil(cv, 999)[0]).max() < 40, 'leeres Bild darf nicht aufgehellt werden'
    blend(cv, np.full((40, 40), 0.5, np.float32), 100, 100, 4)
    assert np.asarray(to_pil(cv, 999)[0]).max() > 100
    print('selfcheck ok')


_selfcheck()

selfcheck ok


In [8]:
%matplotlib widget
# --- GUI ---
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector

S = {'ct': None, 'roi': None, 'crop': None, 'obj': None, 'objname': '-', 'sel': None,
     'items': [], 'act': None, 'drag': None, 'rects': [], 'canvas': None, 'bgimg': None,
     'bgname': 'leeres Bild', 'bg': None, 'bgkey': None, 'sync': False}
SL = w.Layout(width='420px')
ST = {'description_width': '150px'}
FRAME = w.Layout(border='1px solid #555', margin='4px')
ROW = w.Layout(flex_flow='row wrap', align_items='flex-start')
VIEW = 720        # Kantenlaenge der interaktiven Bildflaeche in px - hier aendern, wenn zu klein
DISP = VIEW       # so viele Pixel werden serverseitig gerendert


def fig_widget(fig):
    '''ipympl-Canvas auf feste Groesse nageln. Ohne explizite Hoehe und ohne fixiertes flex
    quetscht die Flexbox die Canvas platt - dann ist keine Flaeche mehr zum Ziehen da.'''
    c = fig.canvas
    c.header_visible = False
    c.footer_visible = False
    c.toolbar_visible = True
    c.resizable = False
    c.layout.width = f'{VIEW}px'
    c.layout.height = f'{VIEW + 44}px'        # Platz fuer die Toolbar
    c.layout.min_width = f'{VIEW}px'
    c.layout.min_height = f'{VIEW + 44}px'
    c.layout.flex = '0 0 auto'
    c.layout.margin = '4px'
    return c


def upload_bytes(widget):
    'FileUpload liefert je nach ipywidgets-Version dict (v7) oder tuple (v8)'
    v = widget.value
    if not v:
        return None, None
    item = list(v.values())[0] if isinstance(v, dict) else v[0]
    return bytes(item['content']), item.get('name', 'upload')


wnd = w.FloatRangeSlider(value=[0.0, 1.0], min=0, max=1, step=0.005, readout_format='.3f',
                         description='Grauwert-Fenster', continuous_update=False,
                         layout=SL, style=ST)
btn_auto = w.Button(description='Auto-Fenster', icon='adjust')
um = w.FloatText(value=30.0, description='um / Pixel:', layout=w.Layout(width='190px'),
                 style={'description_width': '90px'})

# ---------- 1: Ausschneiden ----------
src_path = w.Text(description='Datei:', placeholder='z.B. demo_ct/schraube_3000px_10um.png',
                  layout=w.Layout(width='430px'), style={'description_width': '55px'})
btn_load = w.Button(description='Laden', icon='folder-open')
btn_demo = w.Button(description='Demo-CT Schraube (3000 px, 10 um)', icon='magic',
                    layout=w.Layout(width='270px'))
up = w.FileUpload(accept='.tif,.tiff,.png,.jpg,.jpeg,.bmp', multiple=False, description='Upload')
btn_full = w.Button(description='ganzes Bild', icon='expand')
name = w.Text(value='objekt_01', description='Name:', layout=w.Layout(width='250px'),
              style={'description_width': '55px'})
btn_save = w.Button(description='Ausschnitt exportieren', button_style='success', icon='save')
btn_use = w.Button(description='als Objekt uebernehmen', button_style='info', icon='arrow-down')
img_crop = w.Image(layout=FRAME)
info1 = w.HTML()
log1 = w.HTML()

plt.ioff()                                   # Figures nur als Widget, kein Auto-Output
figc, axc = plt.subplots(figsize=(VIEW / 100, VIEW / 100), dpi=100)
figp, axp = plt.subplots(figsize=(VIEW / 100, VIEW / 100), dpi=100)
for f in (figc, figp):
    f.subplots_adjust(0, 0, 1, 1)
canvas_cut, canvas_scene = fig_widget(figc), fig_widget(figp)
plt.ion()


def on_select(eclick, erelease):
    'RectangleSelector: Rahmen mit der Maus aufgezogen'
    if S['ct'] is None or eclick.xdata is None or erelease.xdata is None:
        return
    set_roi((eclick.xdata, eclick.ydata, erelease.xdata, erelease.ydata))


def show_ct():
    'CT anzeigen (heruntergerechnet), Achsen bleiben in Original-Pixelkoordinaten'
    a = S['ct']
    disp, _ = to_pil(a, DISP, *wnd.value)
    axc.clear()
    axc.imshow(np.asarray(disp)[:, :, 0], cmap='gray', vmin=0, vmax=255,
               extent=(0, a.shape[1], a.shape[0], 0), interpolation='nearest')
    axc.set_axis_off()
    # ponytail: useblit=False - mit blit zeichnet ipympl den Rahmen beim Ziehen teils nicht
    S['sel'] = RectangleSelector(
        axc, on_select, useblit=False, button=[1], interactive=True,
        minspanx=3, minspany=3, spancoords='data',
        props=dict(facecolor='none', edgecolor='#ff5a46', linewidth=1.8),
        handle_props=dict(marker='s', markersize=11, markerfacecolor='#ffd23f',
                          markeredgecolor='black', markeredgewidth=1.0, alpha=1.0))
    figc.canvas.draw_idle()


def set_roi(roi, sync_box=False):
    'einzige Quelle der Wahrheit fuer den Ausschnitt'
    if S['ct'] is None or roi is None:
        return
    x0, y0, x1, y1 = clamp_roi(roi, S['ct'].shape)
    S['roi'] = (x0, y0, x1, y1)
    S['crop'] = S['ct'][y0:y1, x0:x1]
    img_crop.value = png(S['crop'], 260, (), *wnd.value)
    if sync_box and S['sel'] is not None:
        S['sel'].extents = (x0, x1, y0, y1)
        figc.canvas.draw_idle()
    cw, ch = x1 - x0, y1 - y0
    info1.value = (f'<b>CT:</b> {S["ct"].shape[1]} x {S["ct"].shape[0]} px &nbsp;&nbsp; '
                   f'<b>Ausschnitt:</b> {cw} x {ch} px = {cw * um.value:.0f} x {ch * um.value:.0f} um '
                   f'= {cw * um.value / 1000:.3f} x {ch * um.value / 1000:.3f} mm &nbsp;&nbsp; '
                   f'<b>ROI:</b> x {x0}&ndash;{x1}, y {y0}&ndash;{y1}')


def auto_window():
    'Fenster aus dem CT bestimmen - nie aus der Szene, die ist fast nur Hintergrund'
    if S['ct'] is None:
        return
    lo, hi = (float(v) for v in np.percentile(S['ct'], [0.2, 99.9]))
    wnd.value = [max(0.0, lo), min(1.0, max(hi, lo + 0.02))]


def set_ct(a, src=''):
    S['ct'] = a
    log1.value = f'geladen: {src} &ndash; {a.shape[1]} x {a.shape[0]} px'
    auto_window()
    show_ct()
    n = a.shape[0]
    set_roi(S['roi'] or (n // 2 - n // 8, n // 2 - n // 8, n // 2 + n // 8, n // 2 + n // 8),
            sync_box=True)


def demo_roi(n, um_px, idx=3):
    '''Startrahmen um eine Kalibrierkugel (idx 0..3 = 50/100/200/400 um).
    Bewusst gross: bei einem winzigen Rahmen liegen Eck- und Mittelgriff uebereinander,
    dann verschiebt jeder Zug den Rahmen statt ihn aufzuziehen.'''
    cx, cy, dia = DEMO_BALLS[idx]
    jx, jy = mm2px(cx, cy, n, um_px)
    r = max(n // 8, int(round(dia / um_px)))
    return (jx - r, jy - r, jx + r, jy + r)


def load_demo(_):
    um.value = 10.0                     # Observer zieht die um-Angaben nach
    a = demo_ct(3000, 10.0)
    S['roi'] = demo_roi(3000, 10.0, 3)
    set_ct(a, 'Demo-Schraube 10 um/px')


def on_load(_):
    p = src_path.value.strip()
    if not p:
        log1.value = 'kein Pfad angegeben'
        return
    try:
        S['roi'] = None
        set_ct(load_gray(p), p)
    except Exception as e:
        log1.value = f'Fehler: {e}'


def on_upload(ch):
    data, nm = upload_bytes(up)
    if data is None:
        return
    S['roi'] = None
    set_ct(load_gray(data), nm)


def on_save(_):
    if S['crop'] is None:
        return
    p = (PATCHES / (name.value.strip() or 'objekt')).with_suffix('.png')
    save16(S['crop'], p)
    refresh_patches()
    if p.name in list(pick.options):
        pick.value = p.name
    ch, cw = S['crop'].shape
    log1.value = f'gespeichert: {p} ({cw} x {ch} px, 16 bit)'


def on_use(_):
    if S['crop'] is None:
        return
    S['obj'] = S['crop']
    S['objname'] = name.value.strip() or 'Ausschnitt'
    log1.value = ('Ausschnitt ist jetzt das aktive Objekt &ndash; unten ins Bild klicken legt es '
                  'ab, mehrfach klicken legt mehrere ab')
    render()


btn_load.on_click(on_load)
btn_demo.on_click(load_demo)
btn_auto.on_click(lambda _: auto_window())
btn_full.on_click(lambda _: set_roi((0, 0, S['ct'].shape[1], S['ct'].shape[0]), sync_box=True))
btn_save.on_click(on_save)
btn_use.on_click(on_use)
up.observe(on_upload, 'value')
def on_um(ch):
    'um/Pixel geaendert: die px bleiben, alle um-Angaben werden neu gerechnet'
    if S['sync']:
        return
    S['sync'] = True
    oum.value = round(osize.value * um.value, 1)
    S['sync'] = False
    set_roi(S['roi'])
    render()


um.observe(on_um, 'value')

sec1 = w.VBox([
    w.HTML('<h4>1 &ndash; CT laden, Rahmen mit der Maus um das Objekt ziehen</h4>'
           '<div>An den <b>gelben Griffen</b> an Ecken und Kanten ziehen = Rahmen groesser oder '
           'kleiner. In der Mitte anfassen = verschieben. <b>Ausserhalb</b> des Rahmens '
           'anfangen = neuen Rahmen aufziehen. Ist in der Toolbar Zoom oder Pan aktiv, geht '
           'das Ziehen dorthin &ndash; Werkzeug dann abwaehlen.</div>'),
    w.HBox([src_path, btn_load, btn_demo, up], layout=ROW),
    w.HBox([wnd, btn_auto, btn_full, um], layout=ROW),
    info1,
    w.HBox([canvas_cut,
            w.VBox([w.HTML('<b>Ausschnitt</b>'), img_crop, w.HBox([name, btn_save], layout=ROW),
                    btn_use, log1], layout=w.Layout(width='430px', flex='0 0 auto'))], layout=ROW),
])

# ---------- 2: Platzieren (Drag & Drop) ----------
pick = w.Dropdown(description='Objekt:', options=[], layout=w.Layout(width='300px'),
                  style={'description_width': '65px'})
btn_ref = w.Button(description='Liste neu', icon='refresh')
bg_path = w.Text(description='Bild:', placeholder='eigenes Hintergrundbild, z.B. leeraufnahme.tif',
                 layout=w.Layout(width='320px'), style={'description_width': '45px'})
btn_bgload = w.Button(description='laden', icon='folder-open')
bg_up = w.FileUpload(accept='.tif,.tiff,.png,.jpg,.jpeg,.bmp', multiple=False, description='Upload')
btn_bgblank = w.Button(description='leeres Bild verwenden', icon='eraser',
                       layout=w.Layout(width='210px'))
csize = w.IntSlider(value=1024, min=256, max=4096, step=64, description='leeres Bild [px]',
                    continuous_update=False, layout=SL, style=ST)
lvl = w.FloatSlider(value=0.003, min=0, max=0.6, step=0.001, readout_format='.4f',
                    description='Hintergrund-Grauwert', continuous_update=False, layout=SL, style=ST)
noi = w.FloatSlider(value=0.004, min=0, max=0.05, step=0.001, readout_format='.4f',
                    description='Hintergrund-Rauschen', continuous_update=False, layout=SL, style=ST)
btn_bgfit = w.Button(description='Hintergrund an Objekt anpassen', icon='magic',
                     layout=w.Layout(width='250px'))
osize = w.IntSlider(value=120, min=8, max=1024, step=1, description='Groesse [px]',
                    continuous_update=False, layout=SL, style=ST)
oum = w.FloatText(value=1200.0, description='= [um]:', layout=w.Layout(width='190px'),
                  style={'description_width': '60px'})
fea = w.IntSlider(value=8, min=0, max=200, description='Feather-Kante [px]',
                  continuous_update=False, layout=SL, style=ST)
gai = w.FloatSlider(value=1.0, min=0.1, max=2.5, step=0.05, description='Kontrast (Gain)',
                    continuous_update=False, layout=SL, style=ST)
mode = w.Dropdown(description='Einblendung:', layout=w.Layout(width='420px'), style=ST,
                  options=[('Grauwerte behalten (Objekt sichtbar)', False),
                           ('Rand an Hintergrund anpassen (nahtlos)', True)], value=False)
ell = w.Checkbox(value=False, description='Ellipsen-Maske (runder Rand)', indent=False)
shb = w.Checkbox(value=True, description='Rahmen der Objekte zeigen', indent=False)
place_new = w.Checkbox(value=True, description='Klick auf freie Flaeche legt neues Objekt ab',
                       indent=False)
btn_add = w.Button(description='Objekt in die Mitte', icon='plus')
btn_del = w.Button(description='aktives loeschen', icon='times')
btn_clear = w.Button(description='alle loeschen', icon='trash')
sname = w.Text(value='szene_01', description='Datei:', layout=w.Layout(width='250px'),
               style={'description_width': '55px'})
btn_exp = w.Button(description='Bild exportieren (16 bit PNG)', button_style='success', icon='save')
objlist = w.HTML()
info2 = w.HTML()
log2 = w.HTML()


def refresh_patches(*_):
    opts = sorted(p.name for p in PATCHES.glob('*.png'))
    cur = pick.value
    pick.options = opts
    if cur in opts:
        pick.value = cur


def on_pick(ch):
    if pick.value:
        S['obj'] = load_gray(PATCHES / pick.value)
        S['objname'] = pick.value
        log2.value = f'aktives Objekt: {pick.value} &ndash; ins Bild klicken legt es ab'


def canvas_shape():
    'Groesse der Szene: eigenes Hintergrundbild 1:1, sonst das leere Bild'
    return S['bgimg'].shape if S['bgimg'] is not None else (csize.value, csize.value)


def bg_base():
    'Hintergrund liefern: eigenes Bild oder leeres Detektorbild (gecacht)'
    if S['bgimg'] is not None:
        return S['bgimg'].copy()
    key = (csize.value, lvl.value, noi.value)
    if S['bgkey'] != key:
        S['bg'] = make_canvas(csize.value, lvl.value, noi.value)
        S['bgkey'] = key
    return S['bg'].copy()


def set_bg(a, nm):
    'eigenes Hintergrundbild uebernehmen - Pixel bleiben 1:1, damit um/Pixel weiter stimmt'
    S['bgimg'] = a
    S['bgname'] = nm
    h, wd = a.shape
    osize.max = max(h, wd)
    for it in S['items']:                       # Objekte im neuen Bild halten
        it['x'], it['y'] = min(it['x'], wd), min(it['y'], h)
        it['size'] = min(it['size'], max(h, wd))
    log2.value = f'Hintergrundbild: {nm} ({wd} x {h} px)'
    render()


def on_bgload(_):
    p = bg_path.value.strip()
    if not p:
        log2.value = 'kein Pfad angegeben'
        return
    try:
        set_bg(load_gray(p), p)
    except Exception as e:
        log2.value = f'Fehler: {e}'


def on_bgupload(ch):
    data, nm = upload_bytes(bg_up)
    if data is not None:
        set_bg(load_gray(data), nm)


def on_bgblank(_):
    S['bgimg'] = None
    S['bgname'] = 'leeres Bild'
    osize.max = csize.value
    log2.value = 'leeres Bild wird verwendet'
    render()


def active():
    i = S['act']
    return S['items'][i] if i is not None and 0 <= i < len(S['items']) else None


def item_patch(it):
    'skalierte Fassung cachen - sonst wird bei jedem Redraw neu interpoliert'
    if it.get('_q') is None or it['_qs'] != it['size']:
        it['_q'] = resize(it['src'], it['size'])
        it['_qs'] = it['size']
    return it['_q']


def item_box(it):
    'bbox wie blend() sie setzt'
    h, wd = item_patch(it).shape
    x0, y0 = it['x'] - wd // 2, it['y'] - h // 2
    return (x0, y0, x0 + wd, y0 + h)


def new_item(x, y):
    return dict(src=S['obj'], name=S['objname'], size=min(osize.value, max(canvas_shape())),
                x=int(x), y=int(y), f=fea.value, m=mode.value, g=gai.value, e=ell.value,
                _q=None, _qs=-1)


def compose():
    cv = bg_base()
    for it in S['items']:
        blend(cv, item_patch(it), it['x'], it['y'], it['f'], it['m'], it['g'], it['e'])
    return cv


def render(*_):
    cv = compose()
    S['canvas'] = cv
    h, wd = cv.shape
    disp, _ = to_pil(cv, DISP, *wnd.value)
    axp.clear()
    axp.imshow(np.asarray(disp)[:, :, 0], cmap='gray', vmin=0, vmax=255,
               extent=(0, wd, h, 0), interpolation='nearest')
    axp.set_axis_off()
    S['rects'] = []
    for i, it in enumerate(S['items']):
        x0, y0, x1, y1 = item_box(it)
        act = (i == S['act'])
        r = plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                          ec='#ffd23f' if act else '#ff5a46', lw=1.8 if act else 1.0,
                          visible=shb.value)
        axp.add_patch(r)
        S['rects'].append(r)
    figp.canvas.draw_idle()
    it = active()
    if it is None:
        det = '-'
    else:
        c = float(item_patch(it).max()) - lvl.value
        det = (f'{it["name"]}, {it["size"]} px = {it["size"] * um.value:.0f} um, '
               f'Kontrast {c:.3f}, SNR {c / max(noi.value, 1e-6):.1f}')
    info2.value = (f'<b>Bild:</b> {wd} x {h} px = {wd * um.value / 1000:.2f} x '
                   f'{h * um.value / 1000:.2f} mm ({S["bgname"]}) &nbsp;&nbsp; '
                   f'<b>Objekte:</b> {len(S["items"])} &nbsp;&nbsp; '
                   f'<b>aktiv:</b> {det} &nbsp;&nbsp; <b>zum Ablegen:</b> {S["objname"]}')
    objlist.value = '<br>'.join(
        f'{"&#9654;" if i == S["act"] else "&nbsp;&nbsp;"} {i + 1}. {it["name"]} &ndash; '
        f'{it["size"]} px = {it["size"] * um.value:.0f} um @ ({it["x"]}, {it["y"]})'
        for i, it in enumerate(S['items'])) or '<i>noch nichts abgelegt &ndash; ins Bild klicken</i>'


def sync_from_item():
    'Regler auf das aktive Objekt setzen, ohne dabei Aenderungen auszuloesen'
    it = active()
    if it is None:
        return
    S['sync'] = True
    osize.value, fea.value, gai.value, mode.value, ell.value = \
        it['size'], it['f'], it['g'], it['m'], it['e']
    oum.value = round(it['size'] * um.value, 1)
    S['sync'] = False


def apply_to_item(*_):
    if S['sync']:
        return
    it = active()
    if it is not None:
        it.update(size=min(osize.value, max(canvas_shape())), f=fea.value, g=gai.value,
                  m=mode.value, e=ell.value)
    render()


def on_osize(ch):
    if S['sync']:
        return
    S['sync'] = True
    oum.value = round(osize.value * um.value, 1)
    S['sync'] = False
    apply_to_item()


def on_oum(ch):
    if S['sync']:
        return
    S['sync'] = True
    osize.value = int(np.clip(round(oum.value / max(um.value, 1e-6)), osize.min, osize.max))
    S['sync'] = False
    apply_to_item()


def on_press(ev):
    if ev.inaxes is not axp or ev.button != 1 or ev.xdata is None:
        return
    x, y = int(ev.xdata), int(ev.ydata)
    hit = None
    for i in range(len(S['items']) - 1, -1, -1):          # oberstes Objekt gewinnt
        x0, y0, x1, y1 = item_box(S['items'][i])
        if x0 <= x <= x1 and y0 <= y <= y1:
            hit = i
            break
    if hit is None:
        if not place_new.value or S['obj'] is None:
            return
        S['items'].append(new_item(x, y))
        hit = len(S['items']) - 1
    S['act'] = hit
    it = S['items'][hit]
    S['drag'] = (hit, x - it['x'], y - it['y'])
    sync_from_item()
    render()


def on_move(ev):
    'waehrend des Ziehens nur den Rahmen bewegen - neu komponiert wird beim Loslassen'
    if S['drag'] is None or ev.inaxes is not axp or ev.xdata is None:
        return
    i, ox, oy = S['drag']
    it = S['items'][i]
    it['x'], it['y'] = int(ev.xdata) - ox, int(ev.ydata) - oy
    if i < len(S['rects']):
        x0, y0, x1, y1 = item_box(it)
        S['rects'][i].set_bounds(x0, y0, x1 - x0, y1 - y0)
        figp.canvas.draw_idle()


def on_release(ev):
    if S['drag'] is not None:
        S['drag'] = None
        render()


def on_bgfit(_):
    '''Hintergrundpegel und -rauschen auf den Rand des Objekts setzen. Sonst sieht man das
    Patch-Quadrat als hellen/dunklen Kasten, weil Luftpegel im CT und Canvas nicht passen.'''
    it = active()
    src = it['src'] if it is not None else S['obj']
    if src is None or min(src.shape) < 4:
        return
    ring = np.concatenate([src[0], src[-1], src[1:-1, 0], src[1:-1, -1]])
    lvl.value = float(np.clip(ring.mean(), lvl.min, lvl.max))
    noi.value = float(np.clip(ring.std(), noi.min, noi.max))
    log2.value = f'Hintergrund auf Objektrand gesetzt: {lvl.value:.4f} +- {noi.value:.4f}'


def on_add(_):
    if S['obj'] is None:
        log2.value = 'kein Objekt gewaehlt (oben Ausschnitt uebernehmen oder Liste benutzen)'
        return
    h, wd = canvas_shape()
    S['items'].append(new_item(wd // 2, h // 2))
    S['act'] = len(S['items']) - 1
    sync_from_item()
    render()


def on_del(_):
    if active() is not None:
        S['items'].pop(S['act'])
        S['act'] = None
        render()


def on_clear(_):
    S['items'].clear()
    S['act'] = None
    render()


def on_csize(ch):
    if S['bgimg'] is not None:                # eigenes Hintergrundbild gibt die Groesse vor
        return
    n = ch['new']
    osize.max = n
    for it in S['items']:                     # Objekte im Bild halten
        it['x'], it['y'] = min(it['x'], n), min(it['y'], n)
        it['size'] = min(it['size'], n)


def on_export(_):
    cv = compose()
    stem = sname.value.strip() or 'szene'
    p = SCENES / (stem + '.png')
    save16(cv, p)
    log2.value = (f'gespeichert: {p} ({cv.shape[1]} x {cv.shape[0]} px, 16 bit, '
                  f'{um.value:.1f} um/px, {len(S["items"])} Objekte)')


csize.observe(on_csize, 'value')              # zuerst Grenzen anpassen, dann rendern
for wg in (csize, lvl, noi, shb):
    wg.observe(render, 'value')
wnd.observe(lambda ch: (show_ct(), set_roi(S['roi'], sync_box=True), render()), 'value')
osize.observe(on_osize, 'value')
oum.observe(on_oum, 'value')
for wg in (fea, gai, mode, ell):
    wg.observe(apply_to_item, 'value')
pick.observe(on_pick, 'value')
bg_up.observe(on_bgupload, 'value')
btn_ref.on_click(refresh_patches)
btn_bgload.on_click(on_bgload)
btn_bgblank.on_click(on_bgblank)
btn_bgfit.on_click(on_bgfit)
btn_add.on_click(on_add)
btn_del.on_click(on_del)
btn_clear.on_click(on_clear)
btn_exp.on_click(on_export)
figp.canvas.mpl_connect('button_press_event', on_press)
figp.canvas.mpl_connect('motion_notify_event', on_move)
figp.canvas.mpl_connect('button_release_event', on_release)

tune = w.Accordion(children=[
    w.VBox([w.HTML('<i>Uebergang zum Hintergrund</i>'), fea, gai, mode, ell]),
    w.VBox([w.HTML('<i>nur wirksam ohne eigenes Hintergrundbild</i>'),
            csize, lvl, noi, btn_bgfit]),
    w.VBox([place_new, shb]),
], selected_index=None, layout=w.Layout(width='440px'))
tune.set_title(0, 'Einblendung feinjustieren')
tune.set_title(1, 'Leeres Bild einstellen')
tune.set_title(2, 'Verhalten')

controls = w.VBox([
    w.HTML('<b>1. Hintergrund</b> &ndash; leeres Bild oder eigenes Bild laden'),
    w.HBox([bg_path, btn_bgload], layout=ROW),
    w.HBox([bg_up, btn_bgblank], layout=ROW),
    w.HTML('<b>2. Objekt waehlen</b> und ins Bild klicken'),
    w.HBox([pick, btn_ref], layout=ROW),
    w.HBox([osize, oum], layout=ROW),
    w.HBox([btn_add, btn_del, btn_clear], layout=ROW),
    w.HTML('<b>Objekte im Bild</b>'), objlist,
    tune,
    w.HTML('<b>3. Ergebnis speichern</b>'),
    w.HBox([sname, btn_exp], layout=ROW), log2],
    layout=w.Layout(width='460px', flex='0 0 auto'))
sec2 = w.VBox([
    w.HTML('<h4>2 &ndash; Objekte per Maus ins Bild legen und verschieben</h4>'
           '<div>Klick auf freie Flaeche legt das aktive Objekt dort ab &ndash; beliebig oft und '
           'auch mit verschiedenen Objekten aus der Liste. Klick auf ein Objekt greift es, Ziehen '
           'verschiebt es. Das leere Bild ist nur der Platzhalter: oben laesst sich jederzeit ein '
           'eigenes Hintergrundbild laden.</div>'),
    info2,
    w.HBox([controls, canvas_scene], layout=ROW),
])

refresh_patches()
load_demo(None)
S['obj'] = S['crop']
S['objname'] = 'Kugel 400 um'
render()
# JupyterLab packt lange Ausgaben sonst in eine kleine Scrollbox
display(w.HTML('<style>.jp-Cell.jp-mod-outputsScrolled .jp-OutputArea-child'
               '{max-height:none !important;} .jp-OutputArea-child'
               '{max-height:none !important;overflow:visible !important;}</style>'))
display(w.VBox([sec1, w.HTML('<hr>'), sec2]))

HTML(value='<style>.jp-Cell.jp-mod-outputsScrolled .jp-OutputArea-child{max-height:none !important;} .jp-Outpu…